## Tests — OpenAI API Connection (Categorize + Coach)

Live connection checks for every project path that calls OpenAI:

1. **Categorize** — `data_processing.categorize_core.make_openai_llm_call`
2. **Coach** — `model.coach_core.make_openai_chat_call`

Each section makes **one tiny** request. Requires project-root `.env`:
`LLM_API_KEY` (or `OPENAI_API_KEY`), `LLM_MODEL`, and `OPENAI_BASE_URL` / `LLM_API_BASE` if using a gateway.


## Imports + env

In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

from dotenv import load_dotenv


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "data").is_dir() and (candidate / "artifacts").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate project root containing data/ and artifacts/")


ROOT = find_project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

load_dotenv(ROOT / ".env", override=True)

API_KEY = (os.environ.get("LLM_API_KEY") or os.environ.get("OPENAI_API_KEY") or "").strip()
LLM_MODEL = os.environ.get("LLM_MODEL", "gpt-5.2")
BASE_URL = os.environ.get("OPENAI_BASE_URL") or os.environ.get("LLM_API_BASE")

assert API_KEY, (
    f"Missing LLM_API_KEY / OPENAI_API_KEY after loading {(ROOT / '.env')}"
)

from data_processing.categorize_core import make_openai_llm_call
from model.coach_core import make_openai_chat_call

print("Project root:", ROOT)
print("Model:", LLM_MODEL)
print("API key present:", True, f"(len={len(API_KEY)})")
print("Base URL:", BASE_URL or "(default OpenAI)")


## 1) Categorize connection (`make_openai_llm_call`)

One batched categorization-shaped request with a single fake merchant profile.


In [ ]:
try:
    categorize_llm = make_openai_llm_call(model=LLM_MODEL)
    probe_tx = [
        {
            "id": "conn-probe-1",
            "amount_usd": 12.34,
            "mcc_code": "5812",
            "mcc_description": "Eating Places and Restaurants",
            "merchant_city": "Dallas",
            "merchant_state": "TX",
        }
    ]
    results = categorize_llm(probe_tx)
except Exception as exc:  # noqa: BLE001
    msg = str(exc)
    if "429" in msg or "quota" in msg.lower():
        raise RuntimeError(
            "Categorize connection hit quota/rate limit (HTTP 429). "
            "Auth/base URL work, but this gateway has no remaining quota. "
            f"Details: {exc}"
        ) from exc
    raise

assert isinstance(results, list) and len(results) >= 1, results
assert "category" in results[0], results[0]

print("Categorize connection: OK")
print("Sample result:", results[0])


## 2) Coach connection (`make_openai_chat_call`)

One tiny chat completion through the coach client factory.


In [ ]:
try:
    coach_llm = make_openai_chat_call(model=LLM_MODEL)
    reply = coach_llm(
        [
            {"role": "system", "content": "Reply with exactly the word: ok"},
            {"role": "user", "content": "ping"},
        ]
    )
except Exception as exc:  # noqa: BLE001
    msg = str(exc)
    if "429" in msg or "quota" in msg.lower():
        raise RuntimeError(
            "Coach connection hit quota/rate limit (HTTP 429). "
            "Auth/base URL work, but this gateway has no remaining quota. "
            f"Details: {exc}"
        ) from exc
    raise

assert isinstance(reply, str) and reply.strip(), repr(reply)

print("Coach connection: OK")
print("Reply:", repr(reply))
print("LLM connection tests: PASS")
